# autoresearch-problems — Quickstart

This notebook demonstrates the core functionality of the `autoresearch-problems` library:
a framework-agnostic collection of benchmark problems for LLM-driven automated research.

**Contents:**
1. [Browsing the catalog](#1.-Browsing-the-Catalog)
2. [Loading a problem](#2.-Loading-a-Problem)
3. [Inspecting a problem spec](#3.-Inspecting-a-ProblemSpec)
4. [Evaluating outputs](#4.-Evaluating-Outputs)
5. [Using SubprocessRunner](#5.-Using-the-SubprocessRunner)
6. [End-to-end flow](#6.-End-to-End-Flow)
7. [Adding a custom problem](#7.-Adding-a-Custom-Problem)


## Setup

Install the package (skip if already installed):
```bash
pip install autoresearch-problems
```

In [1]:
from autoresearch_problems import registry, run_evaluation, EvalResult, ProblemSpec
from autoresearch_problems.program_runners import SubprocessRunner
import numpy as np
print('autoresearch-problems imported successfully')

autoresearch-problems imported successfully


---
## 1. Browsing the Catalog

List all problem categories and individual problems using the registry.

In [2]:
# List all categories
categories = registry.list_categories()
print('Categories:', categories)

Categories: ['analysis', 'combinatorics', 'geometry']


In [ ]:
# List all problems
all_problems = registry.list_problems()
print('All problems:')
for p in all_problems:
    print(' -', p)

In [ ]:
# List problems in a specific category
combinatorics_problems = registry.list_problems(category='combinatorics')
print('Combinatorics problems:', combinatorics_problems)

---
## 2. Loading a Problem

Load a `ProblemSpec` from the registry using its `category/name` identifier.

In [4]:
spec = registry.load('combinatorics/cap_set')
print(type(spec))
print('Name:', spec.name)
print('Category:', spec.category)

<class 'autoresearch_problems.core.spec.ProblemSpec'>
Name: cap_set
Category: combinatorics


---
## 3. Inspecting a ProblemSpec

A `ProblemSpec` is a **pure data container** (frozen dataclass).  It holds the evaluator source code, parameters, metadata and optional hints for LLM frameworks — but it has **no methods that execute code**.

In [ ]:
print('=== ProblemSpec fields ===')
print(f'name:                  {spec.name}')
print(f'category:              {spec.category}')
print(f'output_type:           {spec.output_type}')
print(f'evaluator_entrypoint:  {spec.evaluator_entrypoint}')
print(f'evaluator_dependencies:{spec.evaluator_dependencies}')
print(f'parameters:            {spec.parameters}')
print(f'timeout_seconds:       {spec.timeout_seconds}')
print(f'maximize:              {spec.maximize}')
print(f'known_best_score:      {spec.known_best_score}')
print(f'source:                {spec.source}')
print(f'tags:                  {spec.tags}')

In [ ]:
# Print the initial prompt (suggestion for LLM frameworks)
if spec.initial_prompt:
    print(spec.initial_prompt)

In [ ]:
# Print the evaluator source code
print(spec.evaluator_code)

In [5]:
print(spec.function_name)

solve


---
## 4. Evaluating Outputs

Use `run_evaluation(spec, output)` to score a candidate output.
The function dynamically loads the evaluator, calls it, and returns an `EvalResult`.

> **Note:** `run_evaluation` is the only entry point for evaluation — `ProblemSpec` itself has no `evaluate()` method by design.

In [6]:
# Valid output: a single-vector cap set
output_valid = np.array([[0, 0, 0, 0, 0, 0, 0, 0]])
result = run_evaluation(spec, output_valid)
print(result)

EvalResult(score=1.0, valid=True, execution_time=0.0021108826622366905, error='', metrics={'set_size': 1})


In [7]:
# Invalid output: three vectors forming an arithmetic progression
output_invalid = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [2, 2, 2, 2, 2, 2, 2, 2],
])
result_invalid = run_evaluation(spec, output_invalid)
print(result_invalid)

EvalResult(score=0.0, valid=False, execution_time=0.002928026020526886, error='Arithmetic progression found: [0 0 0 0 0 0 0 0], [1 1 1 1 1 1 1 1], [2 2 2 2 2 2 2 2]', metrics={})


In [ ]:
# run_evaluation always returns EvalResult — never raises
result_error = run_evaluation(spec, 'this is not an array')
print(result_error)

### Kissing Number Problem

Let's also try the kissing number problem — a geometry/analysis problem from AlphaEvolve.

In [ ]:
kissing_spec = registry.load('analysis/kissing_number')
print(kissing_spec.description)
print('Known best score (d=3):', kissing_spec.known_best_score)

In [ ]:
# The initial_program gives us the icosahedron solution (12 spheres in 3D)
ns = {}
exec(compile(kissing_spec.initial_program, '<prog>', 'exec'), ns)
centres = ns['solve']()
print('centres shape:', centres.shape)

kissing_result = run_evaluation(kissing_spec, centres)
print(kissing_result)

---
## 5. Using the SubprocessRunner

The `SubprocessRunner` executes untrusted LLM-generated code in an isolated subprocess and returns the function's output.  It communicates via a temporary pickle file.

This cleanly separates the **execution environment** (where the candidate program runs) from the **evaluation environment** (where the evaluator runs).

In [ ]:
runner = SubprocessRunner(timeout=30.0)

# A simple candidate program
code = '''
import numpy as np

def solve():
    # A trivial one-element cap set
    return np.array([[0, 0, 0, 0, 0, 0, 0, 0]])
'''

# Execute and retrieve the output
output = runner.execute(code, function_name='solve')
print('Subprocess output shape:', output.shape)
print('Type:', type(output))

---
## 6. End-to-End Flow

This is the complete loop: **load problem → run initial program via subprocess → evaluate → show results**.

In [ ]:
# 1. Load the spec
spec = registry.load('combinatorics/cap_set')

# 2. Run the initial program in a subprocess
runner = SubprocessRunner(timeout=spec.timeout_seconds)
try:
    candidate_output = runner.execute(spec.initial_program, function_name='solve')
    print('Candidate output shape:', candidate_output.shape)
except Exception as e:
    print('Execution error:', e)
    candidate_output = None

In [ ]:
# 3. Evaluate the output
if candidate_output is not None:
    result = run_evaluation(spec, candidate_output)
    print('Score:           ', result.score)
    print('Valid:           ', result.valid)
    print('Execution time:  ', f'{result.execution_time:.4f}s')
    print('Metrics:         ', result.metrics)
else:
    print('Skipped evaluation — no output')

---
## 7. Adding a Custom Problem

Adding a new problem is as simple as creating a directory with two required files:
- `spec.yaml` — metadata and evaluator configuration
- `evaluator.py` — standalone Python file returning a plain `dict`

And two optional files:
- `initial_prompt.md` — hint for LLM frameworks
- `initial_program.py` — seed solution

### Example: a custom problem inline (no files needed)

You can also create a `ProblemSpec` directly in Python:

In [ ]:
custom_evaluator = '''
import numpy as np

def evaluate(output, target=10, **kwargs):
    """Score how close a float is to the target value."""
    try:
        val = float(output)
        distance = abs(val - target)
        score = max(0.0, 1.0 - distance / target)
        return {"score": score, "valid": True, "error": "", "metrics": {"distance": distance}}
    except Exception as exc:
        return {"score": 0.0, "valid": False, "error": str(exc), "metrics": {}}
'''

custom_spec = ProblemSpec(
    name='guess_the_number',
    category='custom',
    description='Output a float as close to the target as possible.',
    output_type='float',
    evaluator_code=custom_evaluator,
    evaluator_entrypoint='evaluate',
    evaluator_dependencies=['numpy'],
    parameters={'target': 10},
    maximize=True,
)

# Test it
for guess in [5.0, 9.0, 10.0, 11.0]:
    result = run_evaluation(custom_spec, guess)
    print(f'guess={guess:5.1f}  score={result.score:.3f}  distance={result.metrics["distance"]:.1f}')

---
## 8. Venv Support

The `SubprocessRunner` can target a specific virtual environment or Python binary, so the LLM-generated code runs in its own isolated dependencies while the evaluator stays in this process.


In [ ]:
from autoresearch_problems.program_runners import SubprocessRunner

# Using a venv — the runner resolves {venv}/bin/python automatically
# runner_venv = SubprocessRunner(timeout=30, venv_path='/path/to/my/venv')

# Or with an explicit Python binary
# runner_custom = SubprocessRunner(timeout=30, python_executable='/usr/bin/python3.11')

# Default: uses sys.executable (the current interpreter)
runner = SubprocessRunner(timeout=30)
print('Python used by runner:', runner._python)


---
## 9. Parallel Execution

Use `execute_batch` to run multiple LLM-generated candidates in parallel.  Failed candidates return an `Exception` object instead of raising, so the result list is always the same length as the input.


In [8]:
import numpy as np
from autoresearch_problems.program_runners import SubprocessRunner

runner = SubprocessRunner(timeout=30)

codes = [
    # Candidate 0: trivial single-vector cap set
    'import numpy as np\ndef solve(): return np.array([[0,0,0,0,0,0,0,0]])',
    # Candidate 1: two-vector cap set
    'import numpy as np\ndef solve(): return np.array([[0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0]])',
    # Candidate 2: invalid Python — will return an Exception
    'this is not valid python!!!',
]

results = runner.execute_batch(codes, function_name='solve', max_workers=3)

for i, r in enumerate(results):
    if isinstance(r, Exception):
        print(f'Candidate {i}: FAILED — {r}')
    else:
        print(f'Candidate {i}: output shape = {r.shape}')


Candidate 0: output shape = (1, 8)
Candidate 1: output shape = (2, 8)
Candidate 2: FAILED — Subprocess exited with code 1 and produced no output.
stderr:   File "/tmp/tmpefybckvi/runner_script.py", line 4
    this is not valid python!!!
                      ^^^^^^
SyntaxError: invalid syntax



---
## 10. Parallel Evaluation

Use `run_evaluation_batch` to evaluate multiple outputs against the same spec in parallel.


In [9]:
import numpy as np
from autoresearch_problems import registry, run_evaluation_batch

spec = registry.load('combinatorics/cap_set')

# Three candidate outputs
outputs = [
    np.array([[0, 0, 0, 0, 0, 0, 0, 0]]),                                    # score 1
    np.array([[0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0]]),         # score 2
    np.array([[1, 1, 1, 0, 0, 0, 0, 0], [2, 0, 0, 0, 0, 0, 0, 0], [0, 1, 2, 0, 0, 0, 0, 0]]),  # invalid (AP)
]

results = run_evaluation_batch(spec, outputs, max_workers=4)

for i, r in enumerate(results):
    print(f'Output {i}: score={r.score}, valid={r.valid}, error={r.error!r}')


Output 0: score=1.0, valid=True, error=''
Output 1: score=2.0, valid=True, error=''
Output 2: score=3.0, valid=True, error=''


---
## 11. End-to-End Pipeline

Use `execute_and_evaluate` to execute a candidate and evaluate it in one call, or `execute_and_evaluate_batch` for a whole batch in parallel.


In [ ]:
from autoresearch_problems import (
    registry,
    execute_and_evaluate,
    execute_and_evaluate_batch,
)

spec = registry.load('combinatorics/cap_set')

# Single candidate
code = 'import numpy as np\ndef solve(): return np.array([[0,0,0,0,0,0,0,0]])'
result = execute_and_evaluate(spec, code, function_name='solve')
print(f'Single candidate: score={result.score}, valid={result.valid}')

# Batch of candidates (run in parallel)
codes = [
    'import numpy as np\ndef solve(): return np.array([[0,0,0,0,0,0,0,0]])',
    'import numpy as np\ndef solve(): return np.array([[0,0,0,0,0,0,0,0],[1,0,0,0,0,0,0,0]])',
    'this is not valid python!!!',  # will get score=0, valid=False
]

batch_results = execute_and_evaluate_batch(spec, codes, function_name='solve', max_workers=3)

for i, r in enumerate(batch_results):
    print(f'Candidate {i}: score={r.score}, valid={r.valid}, error={r.error!r}')


---
## Summary

| Concept | How it works |
|---|---|
| `registry.list_categories()` | Discover available problem categories |
| `registry.list_problems()` | List all (or filtered) problems |
| `registry.load(problem_id)` | Load a `ProblemSpec` from disk |
| `ProblemSpec` | Pure data container — evaluator code + metadata, no methods |
| `run_evaluation(spec, output)` | Execute evaluator and return `EvalResult` |
| `SubprocessRunner.execute(code, fn)` | Run LLM-generated code in isolation |
| `EvalResult` | Score, validity, execution time, metrics |

The evaluators in `problems/` are **completely standalone** — they only depend on numpy and can be copied into any other project with zero library imports.
